In [ ]:
!pip install -q ultralytics roboflow gtts

In [ ]:
import os
import zipfile
import yaml
from google.colab import drive
from ultralytics import YOLO

# 1. ربط Google Drive بـ Colab
print("🔗 جاري ربط Google Drive...")
drive.mount('/content/drive')

# 2. تحديد مسار الملف في Google Drive وتفكيك الضغط
zip_path = '/content/drive/MyDrive/Offline Arabic ADAS.yolov11.zip'

if os.path.exists(zip_path):
    print("📦 تم العثور على الملف! جاري تفكيك الضغط خلال ثوانٍ...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print("✅ تم تفكيك الضغط بنجاح في مجلد 'dataset'!")
else:
    raise FileNotFoundError(f"❌ لم يتم العثور على الملف في المسار: {zip_path}\nتأكد أن الملف موجود في الصفحة الرئيسية للـ Drive بنفس الاسم تماماً.")

# 3. إيجاد ملف data.yaml وتحديث المسارات لـ Colab
dataset_path = os.path.abspath('dataset')
yaml_path = None

for root, dirs, files_list in os.walk(dataset_path):
    if 'data.yaml' in files_list:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path:
    print(f"📄 تم إعداد ملف الإعدادات: {yaml_path}")
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # تصحيح المسار الأساسي للداتاسيت
    data_config['path'] = os.path.dirname(yaml_path)
    with open(yaml_path, 'w') as f:
        yaml.dump(data_config, f)
else:
    raise FileNotFoundError("❌ لم يتم العثور على ملف data.yaml داخل الملف المضغوط!")

# 4. بدء تدريب نموذج YOLOv11 Nano
print("🚀 جاري بدء تدريب YOLOv11 Nano...")
custom_model = YOLO("yolo11n.pt")

results = custom_model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    batch=16,
    name="adas_camera_test"
)

🔗 جاري ربط Google Drive...
Mounted at /content/drive
📦 تم العثور على الملف! جاري تفكيك الضغط خلال ثوانٍ...
✅ تم تفكيك الضغط بنجاح في مجلد 'dataset'!
📄 تم إعداد ملف الإعدادات: /content/dataset/data.yaml
🚀 جاري بدء تدريب YOLOv11 Nano...
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, io

In [ ]:
import shutil
import os

# 1. تحديد المسارات
source_dir = '/content/runs/detect/adas_camera_test'
drive_dir = '/content/drive/MyDrive/ADAS_YOLOv11_Model'

# 2. إنشاء المجلد في الدرايف ونقل الملفات
try:
    shutil.copytree(source_dir, drive_dir, dirs_exist_ok=True)
    print("✅ تم حفظ النموذج والأوزان (best.pt) بنجاح في Google Drive!")
    print(f"📁 المسار: {drive_dir}/weights/best.pt")
except Exception as e:
    print(f"❌ حدث خطأ أثناء النسخ: {e}")

✅ تم حفظ النموذج والأوزان (best.pt) بنجاح في Google Drive!
📁 المسار: /content/drive/MyDrive/ADAS_YOLOv11_Model/weights/best.pt


In [ ]:
import io
import time
from PIL import Image
from base64 import b64decode
from ultralytics import YOLO
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output

# تحميل أوزان نموذجك بعد الانتهاء من التدريب
model = YOLO("runs/detect/adas_camera_test/weights/best.pt")

# قاموس التنبيهات الصوتية لجميع الكلاسات الـ 14 الخاصة بمشروعك
TRAFFIC_CLASSES = {
    "no_entry": "تنبيه، ممنوع الدخول",
    "loose_gravel": "انتبه، تساقط صخور أو حصى متطاير",
    "no_parking_waiting": "تنبيه، ممنوع الانتظار أو الوقوف",
    "pedestrian_crossing_sign": "تنبيه، ممر مشاة أمامك",
    "pedestrian": "تنبيه، انتبه للمشاة",
    "speed_30": "تنبيه، السرعة المحددة 30",
    "speed_40": "تنبيه، السرعة المحددة 40",
    "speed_50": "تنبيه، السرعة المحددة 50",
    "speed_60": "تنبيه، السرعة المحددة 60",
    "speed_80": "تنبيه، السرعة المحددة 80",
    "speed_100": "تنبيه، السرعة المحددة 100",
    "stop_sign": "تنبيه، لوحة قف أمامك",
    "speed_bump": "انتبه، مطب صناعي أمامك",
    "traffic_signal": "انتبه، إشارة مرور أمامك"
}

# متغيرات التحكم بتكرار الصوت
last_detected_label = None
repeat_counter = 0
MAX_REPEATS = 3
AUDIO_DURATION = 3

def play_arabic_alert(text):
    """تحويل النص العربي إلى صوت وتشغيله"""
    tts = gTTS(text=text, lang="ar")
    fp = io.BytesIO()
    tts.write_to_fp(fp)
    display(Audio(fp.getvalue(), autoplay=True))

In [ ]:
def run_ai(img_data):
    global last_detected_label, repeat_counter

    # 1. فك ترميز الصورة القادمة من الكاميرا
    binary = b64decode(img_data.split(',')[1])
    img = Image.open(io.BytesIO(binary)).convert("RGB")

    # 2. الاستدلال بواسطة نموذجك الخاص
    results = model.predict(img, conf=0.35, verbose=False)

    current_frame_sign = None
    for r in results:
        for cls in r.boxes.cls:
            label = model.names[int(cls)]
            if label in TRAFFIC_CLASSES:
                current_frame_sign = label
                break

    # 3. منطق التنبيه الصوتي بدون تكرار مزعج
    if current_frame_sign:
        if current_frame_sign == last_detected_label:
            if repeat_counter < MAX_REPEATS:
                print(f"🔊 تكرار التنبيه ({repeat_counter + 1}/{MAX_REPEATS}): {TRAFFIC_CLASSES[current_frame_sign]}")
                play_arabic_alert(TRAFFIC_CLASSES[current_frame_sign])
                repeat_counter += 1
                time.sleep(AUDIO_DURATION)
        else:
            print(f"🚨 كشف عنصر جديد: {TRAFFIC_CLASSES[current_frame_sign]}")
            last_detected_label = current_frame_sign
            repeat_counter = 1
            play_arabic_alert(TRAFFIC_CLASSES[current_frame_sign])
            time.sleep(AUDIO_DURATION)

In [ ]:
def start_camera_stream():
    js = Javascript("""
    async function streamCamera() {
        const video = document.createElement('video');
        video.style.width = '100%';
        document.body.appendChild(video);

        const stream = await navigator.mediaDevices.getUserMedia({
            video: { facingMode: "environment" }
        });
        video.srcObject = stream;
        await video.play();

        const canvas = document.createElement('canvas');
        const ctx = canvas.getContext('2d');

        while (true) {
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            ctx.drawImage(video, 0, 0);
            const data = canvas.toDataURL('image/jpeg', 0.5);
            google.colab.kernel.invokeFunction(
                'notebook.run_ai', [data], {}
            );
            await new Promise(r => setTimeout(r, 1000));
        }
    }
    streamCamera();
    """)
    display(js)

In [ ]:
# ربط دالة Python بكود الـ JavaScript للاتصال الكاميرا
output.register_callback("notebook.run_ai", run_ai)

print("📱 جاري تشغيل الكاميرا... وجّه الجوال نحو الشاشة أو صورة لوحة للتجربة!")
start_camera_stream()